# importlib_metadata Package Testing

This notebook tests the importlib_metadata package, which provides access to package metadata and distribution information. Every cell calls the `importlib_metadata` package itself, not the standard-library `importlib.metadata`.

## Example 1: Basic Package Information

In [ ]:
import importlib.metadata as stdlib_metadata
import importlib_metadata

# The package reports its own version, and the standard library agrees.
version = importlib_metadata.version("importlib_metadata")
print(f"importlib_metadata version: {version}")
assert version == stdlib_metadata.version("importlib_metadata"), version

# The import name maps back to the distribution that installed it.
providers = importlib_metadata.packages_distributions().get("importlib_metadata", [])
print(f"import name 'importlib_metadata' comes from: {providers}")
assert [name.replace("-", "_").lower() for name in providers] == ["importlib_metadata"], providers

# List installed distributions
packages = list(importlib_metadata.distributions())
print(f"Total packages installed: {len(packages)}")
for dist in packages[:5]:
    print(f"{dist.metadata.get('Name')}: {dist.version}")
names = {(dist.metadata.get("Name") or "").replace("-", "_").lower() for dist in packages}
# This kernel runs on ipykernel, so its distribution is always installed.
assert {"importlib_metadata", "ipykernel"} <= names

## Example 2: Package Metadata and Files

In [ ]:
import importlib_metadata

# Get detailed metadata for the package itself
metadata = importlib_metadata.metadata("importlib_metadata")
print(f"Name: {metadata['Name']}")
print(f"Version: {metadata['Version']}")
print(f"Summary: {metadata.get('Summary', 'N/A')}")
print(f"Requires-Python: {metadata.get('Requires-Python', 'N/A')}")
assert metadata["Name"].replace("-", "_").lower() == "importlib_metadata"
assert metadata["Version"] == importlib_metadata.version("importlib_metadata")

# Get the files the distribution installed
files = importlib_metadata.files("importlib_metadata")
assert files, "importlib_metadata lists no installed files"
print(f"\nInstalled files: {len(files)}; first 5:")
for file in files[:5]:
    print(f"  {file}")

# Each entry is a PackagePath that can locate and read the installed file.
init = next(f for f in files if f.parts == ("importlib_metadata", "__init__.py"))
assert init.locate().is_file(), init.locate()
assert "def version(" in init.read_text(encoding="utf-8")
print(f"\n{init} -> {init.locate()}")

## Example 3: Entry Points Discovery

In [ ]:
import importlib_metadata
import pathlib
import string
import tempfile

# A small distribution written to a temporary directory: METADATA plus entry_points.txt.
with tempfile.TemporaryDirectory() as site:
    dist_info = pathlib.Path(site, "demo_plugins-1.0.dist-info")
    dist_info.mkdir()
    (dist_info / "METADATA").write_text(
        "Metadata-Version: 2.1\nName: demo-plugins\nVersion: 1.0\nRequires-Dist: idna>=3\n",
        encoding="utf-8",
    )
    (dist_info / "entry_points.txt").write_text(
        "[console_scripts]\ndemo-tool = json.tool:main\n\n[demo.formatters]\ntitle = string:capwords\n",
        encoding="utf-8",
    )

    found = [d for d in importlib_metadata.distributions(path=[site])]
    assert [d.metadata["Name"] for d in found] == ["demo-plugins"], found
    dist = found[0]
    print(f"Found {dist.metadata['Name']} {dist.version}, requires {dist.requires}")
    assert dist.version == "1.0" and dist.requires == ["idna>=3"]

    # Discover entry points by group, then load one.
    entry_points = dist.entry_points
    print(f"Entry point groups: {sorted(entry_points.groups)}")
    assert entry_points.groups == {"console_scripts", "demo.formatters"}
    (formatter,) = entry_points.select(group="demo.formatters")
    print(f"  {formatter.name} -> {formatter.value}")
    assert (formatter.name, formatter.module, formatter.attr) == ("title", "string", "capwords")
    assert formatter.load() is string.capwords
    assert formatter.load()("hello entry points") == "Hello Entry Points"

# The environment-wide view: every console script of every installed distribution.
console_scripts = importlib_metadata.entry_points(group="console_scripts")
print(f"\nConsole scripts installed: {len(console_scripts)}")
for ep in list(console_scripts)[:5]:
    print(f"  {ep.name} -> {ep.value}")

## Example 4: Distribution Requirements

In [ ]:
import importlib_metadata

# Get requirements for a package that is always installed alongside this kernel
dist = importlib_metadata.distribution("ipykernel")
print(f"Package: {dist.metadata['Name']} v{dist.version}")

requires = dist.requires
assert requires, "ipykernel declares no dependencies"
print(f"\nDependencies ({len(requires)}):")
for req in requires[:10]:  # Show first 10
    print(f"  {req}")
# ipykernel depends on jupyter_client to talk to the notebook front end.
assert any(req.replace("_", "-").lower().startswith("jupyter-client") for req in requires), requires

# Optional features
provides_extra = dist.metadata.get_all("Provides-Extra") or []
print(f"\nOptional features: {provides_extra}")

# A missing distribution raises PackageNotFoundError.
try:
    importlib_metadata.distribution("no-such-distribution-for-this-notebook")
except importlib_metadata.PackageNotFoundError as error:
    print(f"\nPackageNotFoundError as expected: {error}")
else:
    raise AssertionError("a missing distribution was found")

## Example 5: Custom Distribution Analysis

In [ ]:
import importlib_metadata
from collections import defaultdict

# Analyze installed packages by file count and dependencies
def analyze_distributions():
    size_stats = defaultdict(int)
    dep_stats = defaultdict(int)

    for dist in importlib_metadata.distributions():
        name = dist.metadata.get("Name") or "?"
        size_stats[name] = len(dist.files or [])
        dep_stats[name] = len(dist.requires or [])

    # Show packages with most files
    print("Top 5 packages by file count:")
    sorted_by_size = sorted(size_stats.items(), key=lambda x: x[1], reverse=True)
    for name, count in sorted_by_size[:5]:
        print(f"  {name}: {count} files")

    # Show packages with most dependencies
    print("\nTop 5 packages by dependency count:")
    sorted_by_deps = sorted(dep_stats.items(), key=lambda x: x[1], reverse=True)
    for name, count in sorted_by_deps[:5]:
        print(f"  {name}: {count} dependencies")
    return size_stats, dep_stats

size_stats, dep_stats = analyze_distributions()
by_name = {name.replace("-", "_").lower(): count for name, count in size_stats.items()}
assert by_name["importlib_metadata"] > 0, by_name.get("importlib_metadata")
assert max(dep_stats.values()) > 0